In [ ]:
pip install transformers torch


For a single file


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load the IndicBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
model = AutoModel.from_pretrained("ai4bharat/indic-bert")

# Sample Tamil text (from the uploaded file)
tamil_text = """
ஆண்கள் எதைப்பற்றியும் அஞ்சாதவர்கள் ஒன்றைத் தவிர அவர்களின் பயம் எல் லாம்
தனது மரணத்திற்குப் பிறகு தனது ஆலை எவன் போடுவான் என்கிற பயம் மட்டுமே முறைத்த பெண்ணை துரத்தி பிடிப்பவர்கள் ஆண்கள்.
"""

# Tokenize the input text
inputs = tokenizer(tamil_text, return_tensors="pt", truncation=True, padding=True)

# Get the model output (last hidden state)
with torch.no_grad():
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state

# Average the embeddings to get a single vector for the sentence
sentence_embedding = torch.mean(embeddings, dim=1)

# Print the shape of the sentence embedding
print(sentence_embedding.shape)

# Optionally, convert to a numpy array and use for further tasks
embedding_vector = sentence_embedding.numpy()
print(embedding_vector)


In [ ]:
pip install transformers torch python-docx numpy pandas openpyxl


In [ ]:
import os
import zipfile

# Specify the path where the new folder will be created
extract_to_folder = "/content/unzipped_2_files"
os.makedirs(extract_to_folder, exist_ok=True)

# Upload the ZIP file to Colab
from google.colab import files
uploaded = files.upload()

# Get the name of the uploaded ZIP file
zip_file_name = list(uploaded.keys())[0]  # This gets the file name

# Unzip the ZIP file into the new folder
with zipfile.ZipFile(f"/content/{zip_file_name}", 'r') as zip_ref:
    zip_ref.extractall(extract_to_folder)

# Verify the files have been extracted
print(f"Files extracted to: {extract_to_folder}")
print(os.listdir(extract_to_folder))


For all the files using Indic Bert


In [ ]:
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from docx import Document
import numpy as np

# Function to read text from a .docx file
def extract_text_from_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)
    return '\n'.join(full_text)

# Function to generate embeddings for a given text
def get_sentence_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state
    sentence_embedding = torch.mean(embeddings, dim=1)  # Average to get sentence-level embedding
    return sentence_embedding.numpy()

# Function to chunk text if it's too long
def chunk_text(text, max_length=512):
    tokens = tokenizer.encode(text)
    for i in range(0, len(tokens), max_length):
        yield tokenizer.decode(tokens[i:i + max_length])

# Folder path where .docx files are located
folder_path = "/content/unzipped_2_files/Transcript"

# List all .docx files in the folder
docx_files = [f for f in os.listdir(folder_path) if f.endswith('.docx')]

# Load the IndicBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
model = AutoModel.from_pretrained("ai4bharat/indic-bert")

# Data structure to hold the results (file name, embedding)
data = []

# Iterate over all files and process them
for file_name in docx_files:
    file_path = os.path.join(folder_path, file_name)
    # Extract text from the .docx file
    text = extract_text_from_docx(file_path)

    # Handle text chunking
    embeddings = []
    for chunk in chunk_text(text):
        embedding = get_sentence_embedding(chunk, tokenizer, model)
        embeddings.append(embedding)

    # Average embeddings if there are multiple chunks
    if embeddings:
        final_embedding = np.mean(embeddings, axis=0)
        flattened_embedding = final_embedding.flatten().tolist()

        # Append the file name and its corresponding embedding to the data list
        data.append([file_name] + flattened_embedding)

# Check if at least one document was processed to define column headers
if data:
    columns = ['File Name'] + [f'Embedding_{i+1}' for i in range(len(flattened_embedding))]  # Column names
    df = pd.DataFrame(data, columns=columns)

    # Save the DataFrame to an Excel file
    output_file = "/content/embeddings.xlsx"
    df.to_excel(output_file, index=False)

    # Download the generated Excel file
    from google.colab import files
    files.download(output_file)

    print(f"Embeddings for {len(docx_files)} files saved to {output_file} and downloaded.")
else:
    print("No embeddings were processed as no documents were found.")
